# Trabalho de Instalações Elétricas
## Escola Politécnica
## Universidade Federal do Rio de Janeiro
### Eduardo Motta e Anderson
Agosto 2026

# Escopo do Trabalho
Lorem Ipsum

In [10]:
# carrega as bibliotecas
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import scipy as sp
import warnings
import math

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "sans-serif",
    "font.size": 14,
    "font.sans-serif": ["Helvetica", "Arial", "Dejavu Sans"],
})

# Okabe-Ito colorblind-friendly palette
color_blind_palette = [
    (0.902, 0.624, 0.000),   # Orange
    (0.337, 0.706, 0.914),   # Sky Blue
    (0.000, 0.620, 0.451),   # Bluish Green
    (0.941, 0.894, 0.259),   # Yellow
    (0.000, 0.447, 0.698),   # Blue
    (0.835, 0.369, 0.000),   # Vermillion
    (0.800, 0.475, 0.655),   # Reddish Purple
]

# Global matplotlib style settings
plt.rcParams.update({
    'font.family':      'serif',
    'font.size':        14,
    'axes.grid':        True,
    'grid.color':       '#D9D9D9',
    'grid.linestyle':   '--',
    'axes.linewidth':   1.2,
    'figure.figsize':   (7, 14/3),   # ~600px wide, 2/3 aspect
})

# Características Técnicas

In [11]:
V_pn = 127
V_pp = 220

# Criação das Classes

In [12]:
# Classe do Projeto

class ProjetoEletrico:
    def __init__(self):
        self.comodos = []

    def adicionar_comodo(self, comodo):
        """Adiciona o cômodo ao projeto, evitando que o mesmo cômodo seja inserido duas vezes."""
        if comodo not in self.comodos:
            self.comodos.append(comodo)

    def calcular_baricentro(self):
        """Calcula o centro de carga lendo os cômodos dinamicamente no momento da chamada."""
        soma_px = 0.0
        soma_py = 0.0
        potencia_total = 0.0
        
        # Lê as cargas diretamente de cada cômodo, garantindo que não haja acúmulo oculto
        for comodo in self.comodos:
            for c in comodo.cargas_posicionadas:
                soma_px += c.potencia_va * c.x
                soma_py += c.potencia_va * c.y
                potencia_total += c.potencia_va
                
        if potencia_total == 0:
            return 0.0, 0.0
            
        return (soma_px / potencia_total), (soma_py / potencia_total)

    def limpar_comodos(self):
        """Limpa os cômodos já inseridos para evitar acúmulo ao rodar a célula novamente."""
        self.comodos = []

    
#circuito
#recebe cargas já poisicionadas e o projet vai utilizar para alguma coisa
#teste4

# Classe dos Cômodos

class Comodo:
    def __init__(self, nome, tipo, area, perimetro):
        self.nome = nome
        self.tipo = tipo.lower()
        self.area = area
        self.perimetro = perimetro
        
        # Variáveis de previsão (Passo 1)
        self.prev_iluminacao_va = 0
        self.prev_qtd_tugs = 0
        self.prev_potencia_tugs_va = 0
        
        # Cargas definitivas com posição (Passos 2 e 3)
        self.cargas_posicionadas = []

    def prever_cargas_nbr5410(self):
        """Passo 1: Calcula o mínimo exigido pela NBR-5410."""
        # --- ILUMINAÇÃO ---
        if self.area <= 6:
            self.prev_iluminacao_va = 100
        else:
            adicionais = math.floor((self.area - 6) / 4)
            self.prev_iluminacao_va = 100 + (adicionais * 60)
            
        # --- TUGs ---
        if self.tipo in ['cozinha', 'copa', 'área de serviço', 'lavanderia']:
            # 1 tomada a cada 3,5m de perímetro ou fração
            self.prev_qtd_tugs = math.ceil(self.perimetro / 3.5)
            
            # As 3 primeiras recebem 600VA, as demais 100VA
            if self.prev_qtd_tugs <= 3:
                self.prev_potencia_tugs_va = self.prev_qtd_tugs * 600
            else:
                self.prev_potencia_tugs_va = (3 * 600) + ((self.prev_qtd_tugs - 3) * 100)
                
        elif self.tipo == 'banheiro':
            # Mínimo de 1 tomada (600VA) junto ao lavatório
            self.prev_qtd_tugs = 1
            self.prev_potencia_tugs_va = 600

        elif self.tipo == 'escritorio comercial':
            # De acordo com a area util
            if self.area <= 40: # self.area <= 40m^2
                # Calcula os dois valores aplicando o arredondamento para cima (math.ceil)
                tugs_perimetro = math.ceil(self.perimetro / 3)
                tugs_area = math.ceil(self.area / 4)
                
                # Pega o maior valor entre os dois
                if tugs_perimetro > tugs_area: 
                    self.prev_qtd_tugs = tugs_perimetro
                else: 
                    self.prev_qtd_tugs = tugs_area

            else: # 40m^2 < self.area
                # Para o else, a fórmula também tem o arredondamento no final: 10 + [ (A-40)/10 ]
                self.prev_qtd_tugs = 10 + math.ceil((self.area - 40) / 10)

        elif self.tipo == 'loja' or self.tipo == 'salao comercial':
            # Regra: 1 tomada para cada 30 m² de área
            # Denconsidera-se na conta tomadas especificas de vitrines e caixas registradoras
            # Usamos math.ceil para arredondar para cima (ex: 31m² / 30 = 1.03 -> arredonda para 2)
            self.prev_qtd_tugs = math.ceil(self.area / 30)
            
            # Garantir que o mínimo seja 1 (caso a área seja muito pequena ou zero)
            if self.prev_qtd_tugs < 1:
                self.prev_qtd_tugs = 1

            # Potencia minima por tomada e 200
            self.prev_potencia_tugs_va = 200*self.prev_qtd_tugs
                    
        else: 
            # Áreas secas (Salas, quartos, etc.): 1 tomada a cada 5m
            self.prev_qtd_tugs = max(1, math.ceil(self.perimetro / 5))
            self.prev_potencia_tugs_va = self.prev_qtd_tugs * 100
            
        print(f"[{self.nome}] Previsão NBR-5410: {self.prev_iluminacao_va}VA (Luz), {self.prev_qtd_tugs} TUGs ({self.prev_potencia_tugs_va}VA)")

    def adicionar_carga_com_posicao(self, descricao, potencia_va, x, y):
        """Passos 2 e 3: Adiciona qualquer carga já com sua posição X, Y definida."""
        nova_carga = Carga(f"{self.nome} - {descricao}", potencia_va, x, y)
        self.cargas_posicionadas.append(nova_carga)
    
    def limpar_cargas_posicionadas(self):
        """Limpa as cargas já inseridas para evitar acúmulo ao rodar a célula novamente."""
        self.cargas_posicionadas = []

# Classe das Cargas

class Carga:
    def __init__(self, descricao, potencia_va, x, y):
        self.descricao = descricao
        self.potencia_va = potencia_va
        self.x = x
        self.y = y

# Definindo os cômodos e prevendo suas cargas

In [13]:
projeto = ProjetoEletrico()
quarto1 = Comodo("Quarto 01", "quarto", 12.0, 14.0)
cozinha = Comodo("Cozinha", "cozinha", 8.0, 12.0)
barbearia = Comodo("barb","salao comercial",44,32)


quarto1.prever_cargas_nbr5410() 
cozinha.prever_cargas_nbr5410()
barbearia.prever_cargas_nbr5410()

[Quarto 01] Previsão NBR-5410: 160VA (Luz), 3 TUGs (300VA)
[Cozinha] Previsão NBR-5410: 100VA (Luz), 4 TUGs (1900VA)
[barb] Previsão NBR-5410: 640VA (Luz), 2 TUGs (400VA)


## Adicionando cargas específicas e posicionando cargas

In [14]:
# Cargas do Quarto 1
quarto1.limpar_cargas_posicionadas()  # Limpa cargas anteriores antes de adicionar novas

quarto1.adicionar_carga_com_posicao("Ar Condicionado (TUE)", 1200, x=2.5, y=4.0)
quarto1.adicionar_carga_com_posicao("Luz Teto 1", 40, x=2.0, y=2.0)
quarto1.adicionar_carga_com_posicao("Luz Teto 2", 40, x=3.0, y=2.0)
quarto1.adicionar_carga_com_posicao("Luz Teto 3", 40, x=2.0, y=3.0)
quarto1.adicionar_carga_com_posicao("Luz Teto 4", 40, x=3.0, y=3.0)
quarto1.adicionar_carga_com_posicao("TUG 1", 100, x=0.0, y=1.5)
quarto1.adicionar_carga_com_posicao("TUG 2", 100, x=4.0, y=1.5)
quarto1.adicionar_carga_com_posicao("TUG 3", 100, x=2.0, y=4.0)

# Cargas da Cozinha
cozinha.limpar_cargas_posicionadas()  # Limpa cargas anteriores antes de adicionar novas

cozinha.adicionar_carga_com_posicao("Micro-ondas", 1200, x=2.0, y=5.0)
cozinha.adicionar_carga_com_posicao("Luz Teto 1", 50, x=2.0, y=6.0)
cozinha.adicionar_carga_com_posicao("Luz Teto 2", 50, x=3.0, y=6.0)
cozinha.adicionar_carga_com_posicao("TUG 1", 600, x=2.0, y=4.0)
cozinha.adicionar_carga_com_posicao("TUG 2", 600, x=4.0, y=4.0)
cozinha.adicionar_carga_com_posicao("TUG 3", 600, x=2.0, y=8.0)
cozinha.adicionar_carga_com_posicao("TUG 4", 100, x=4.0, y=8.0)


## Calculando baricentro de carga

In [15]:
projeto.limpar_comodos()  # Limpa cômodos anteriores antes de adicionar novos

projeto.adicionar_comodo(quarto1)
projeto.adicionar_comodo(cozinha)

x_qdc, y_qdc = projeto.calcular_baricentro()

print(f"\nPosição ideal do QDC: X = {x_qdc:.2f}, Y = {y_qdc:.2f}")


Posição ideal do QDC: X = 2.44, Y = 4.71
